# Do bounded confidence or a co-evolving network change the controversy-axis result?

Two A/B backtests for [Lightningfish](https://github.com/rajul-kk/LightningFish), run on Kaggle's free T4 GPU. Same setup as `kaggle_controversy.ipynb` (qwen2.5:7b served locally by Ollama, no API key, $0) and the same population size (24 agents, 3-4 rounds), since that's what fits comfortably in a T4's VRAM without babysitting it.

The calibrated controversy run in METHODOLOGY.md found the simulation's crowd-split prediction sitting below chance: 38% against a 53% best baseline, n=74. Two mechanisms got proposed afterward as ways to make the crowd's split more realistic. Bounded confidence (Hegselmann-Krause style) gates T3's herding update on `confidence_bound`, so an agent ignores a target too far from its own opinion instead of being pulled toward it regardless. That one's already been run (see the "bounded confidence" section below), and it doesn't move the needle: 48% with the gate off vs. 52% with it on, both below the 62% baseline, both nowhere near significant (p=0.98 and p=0.92). The 4-point gap turns out to be about a coin flip's worth of per-event churn, not a real effect, and it's logged as such in METHODOLOGY.md's results table.

The second mechanism, a co-evolving follower network, hasn't been tested against real events yet. That's what this notebook adds. Agents drop a followed peer once that peer's opinion drifts too far and refill from someone closer, so echo chambers form dynamically instead of being fixed at round 0 (`rewire_follower_graph`, opt-in via `coevolving_network=True`).

Neither of these is a claim that the mechanism "works." They're mechanism tests. Given every axis on this domain has failed the ladder so far, including bounded confidence, the honest prior going in is that the network arm probably won't move it either. Whatever it does, reporting it plainly is the point of running this at all.


---
## What the data is

Same source as every other HN backtest in this repo: the [Algolia API](https://hn.algolia.com/api), free and unauthenticated at roughly 10k requests/hour, so no key and no scraping needed.

The sample is settled stories at least 24 hours old. `PULL_LIMIT` has to be generous. A local test run at `limit=60` only cleared 20 scoreable events out of 60 (33%), well under the harness's 15/15 minimum split, so this pulls 250 to leave real margin.

Each agent's seed is strictly submission-time fields: title, author and karma, url domain, type, self-text. Never the outcome. That's enforced in code and covered by tests, not just a convention.

The label is `num_comments / points` at settlement: a ratio of 0.7 or higher counts as "contested," under 0.4 is "consensus," and anything in between (or under 20 points) gets skipped as too ambiguous to score. `kaggle_controversy.ipynb` has the fuller rationale for why that particular cutoff.


---
## What's different between the arms

All three runs pull the same events, use the same calibration/evaluation split (a deterministic hash of the event id), and derive their threshold the same way. The only thing that changes is which flag gets passed to `_run_hn_controversy_calibrated`:

| Run | bounded_confidence | coevolving_network |
|---|---|---|
| `hn-controversy-calibrated` | True (default) | False (default) |
| `hn-controversy-calibrated-nobc` | False | False |
| `hn-controversy-calibrated-network` | True (default) | True |

The network arm's control is really just the first row, already run as part of the bounded-confidence test. The harness's cache keys on both flags (`:bc1`, `:net1` suffixes), so this notebook won't re-simulate that arm, only the new network-enabled one.


---
## 1. Setup

Sidebar first: **Accelerator → GPU**, **Internet → On**.

Install `zstd` before Ollama. Its installer needs it to extract, Kaggle's base image doesn't ship it, and skipping this step makes the install fail silently. You won't notice until a few cells later, when it shows up as a confusing `FileNotFoundError: 'ollama'`.


In [ ]:
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd > /dev/null 2>&1
!zstd --version || echo "WARNING: zstd missing - the install below will fail"
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import shutil, subprocess, time, requests

if shutil.which("ollama") is None:
    raise RuntimeError(
        "ollama not found after install. Scroll up for the installer's error: "
        "usually zstd (cell above) or Internet disabled in the sidebar."
    )

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("ollama up"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama installed but the server did not start")


In [ ]:
MODEL = "qwen2.5:7b"

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ollama pull {MODEL}

requests.post("http://localhost:11434/api/generate",
              json={"model": MODEL, "prompt": "hi", "stream": False, "keep_alive": -1},
              timeout=600)

for m in requests.get("http://localhost:11434/api/ps").json().get("models", []):
    vram = m.get("size_vram", 0) / 1e9
    print(f"{m['name']}: {vram:.2f} GB in VRAM")
    assert vram > 0, "model landed on CPU - enable the GPU accelerator, this is pointless otherwise"
print("GPU inference confirmed")


In [ ]:
!git clone --depth 1 https://github.com/rajul-kk/LightningFish.git /kaggle/working/lf
!pip -q install anthropic openai scipy requests pytest praw yfinance

import os, sys
os.chdir("/kaggle/working/lf")
sys.path.insert(0, "/kaggle/working/lf")

# Engine + HN suites only; also confirms bounded confidence, the
# CachingAdapter kwarg-forwarding fix, and the calibrated-threshold code are
# actually present in this clone (all pushed in commit 17bcc07).
!python -m pytest tests/core tests/hn -q 2>&1 | tail -5


---
## 2. Configuration


In [ ]:
PULL_LIMIT = 250      # stories to pull; expect roughly 1 in 3 to be scoreable
N_AGENTS   = 24        # matches kaggle_controversy.ipynb's validated GPU size
N_ROUNDS   = 4

os.environ["LIGHTNINGFISH_MODEL"] = f"ollama:{MODEL}"
os.environ["LIGHTNINGFISH_N_AGENTS"] = str(N_AGENTS)
os.environ["LIGHTNINGFISH_N_ROUNDS"] = str(N_ROUNDS)
os.environ["LIGHTNINGFISH_LOCAL_TIMEOUT"] = "120"
os.environ["PYTHONUNBUFFERED"] = "1"
print(f"{MODEL} | {N_AGENTS} agents x {N_ROUNDS} rounds | pulling {PULL_LIMIT}, two arms")


### Throughput check

Roughly 26 model calls per event, times two arms running back to back. Worth confirming the per-call cost before committing to the full run. Double digits here means you're actually on CPU, whatever the GPU assertion above said.


In [ ]:
from lightningfish_core.llm_provider import make_provider

provider = make_provider(f"ollama:{MODEL}")
t0 = time.time()
for _ in range(3):
    provider.get_opinion("Output ONLY a number between -1 and 1.", "Rate: 0.5", f"ollama:{MODEL}")
per_call = (time.time() - t0) / 3
print(f"{per_call:.2f}s per call  ->  ~{per_call*26:.0f}s per event  ->  ~{per_call*26*2:.0f}s per event-pair (both arms)")


---
## 3. Run all three arms

The first two (`hn-controversy-calibrated`, `hn-controversy-calibrated-nobc`) are the bounded-confidence A/B, already summarized above. The third (`hn-controversy-calibrated-network`) is the new one. It adds the co-evolving-network arm, using the first row as its control.


In [ ]:
!python -m tests.integration.run_backtest hn-controversy-calibrated {PULL_LIMIT} 2>&1 | tee /kaggle/working/bc_on.log


In [ ]:
!python -m tests.integration.run_backtest hn-controversy-calibrated-nobc {PULL_LIMIT} 2>&1 | tee /kaggle/working/bc_off.log


### Co-evolving network arm

Same events, same split, and `bounded_confidence` held at its default (True). The only variable here is whether the follower graph rewires each round.


In [ ]:
!python -m tests.integration.run_backtest hn-controversy-calibrated-network {PULL_LIMIT} 2>&1 | tee /kaggle/working/network_on.log


### Reading the logs

Each log ends with the standard report block (`beats_baselines`, `p_value_vs_best`) and along the way prints how many events survived the controversy-direction filter and how the calibration/evaluation split came out. Those first two numbers should match across all three logs, since they all pull from the same event set; only the calibrated threshold itself is likely to differ slightly run to run.

For bounded confidence (`bc_on.log` vs. `bc_off.log`), the result's already in: a non-effect, discussed above. Re-running it should land close to 48%/52% again if you want to check reproducibility, but at n=42 each event is worth about 2.4 points, so don't expect, or read into, a different verdict.

For the network arm (`network_on.log`, against `bc_on.log` as its control), there are really two ways this goes. If accuracy comes back materially higher, or `beats_baselines` flips to a PASS, that's a real signal worth carrying forward, pending a bigger n to make sure it isn't the same kind of noise bounded confidence turned out to be. If it lands about the same or worse, that's consistent with everything else found on this domain: HN reception seems to be driven by who posts and who replies early, not by which local social mechanic the crowd is running on. Either way, log it in METHODOLOGY.md next to the bounded-confidence rows, and apply the same discipline that caught the bounded-confidence gap being churn: diff the two reports event by event before trusting a raw accuracy difference.


---
## 4. Save


In [ ]:
!cp -r .cache/lightningfish /kaggle/working/cache
!ls -la /kaggle/working/cache


All three logs and the run cache land in `/kaggle/working/`. Grab them from the notebook's Output tab. The cache keys each simulated run by which flags were on (`:bc1`, `:net1` suffixes), so re-scoring these same events later costs nothing.

Whatever the network arm turns up belongs in [METHODOLOGY.md](https://github.com/rajul-kk/LightningFish/blob/main/METHODOLOGY.md) next to the bounded-confidence rows. A negative result is exactly as worth recording as any of the others in that table.
